# InnerSource Synthesis Example <a name="top"></a>

## Table of Contents
1. [Introduction](#introduction)
2. [End-to-end Synthesis Example and Explanations](#example)
    - [Libraries and helper functions](#libraries)
    - [Local paths](#paths)
    - [2.1 Prepare input events](#inputevents)
    - [2.2 Define Synthesis extension classes and Lookup](#extensions)
    - [2.3 Run Synthesis](#runsynthesis)
    - [2.4 Review Synthesis outputs](#output)
3. [Conclusion](#conclusion)

## Introduction <a name="introduction"></a>
Synthesis is the process of reading pinned records, grouping these records (by pin, account, and period as defined by the user), and generating a single record for each group-period. This resultant record comprises fields that are a reconciliation of the values within the specified group-period.

<img src="./example/images/mermaid-diagram-2022-12-22-114414.png" alt="Synthesis example"/>

For a comprehensive understanding, please refer to the [Oxygen Synthesis tutorial.](https://pages.experian.com/display/ODC/Oxygen+tutorials%3A+Hands-on+Synthesis+stage)

[Return to Top](#top)

## End-to-end Synthesis Example and Explanations <a name="example"></a>
The Synthesis job orchestrates through the following steps:

1. [**Prepare input events**](#inputevents)
2. [**Define Synthesis extension classes and Lookup**](#extensions)
3. [**Run Synthesis**](#runsynthesis)
4. [**Review synthesis outputs**](#output)

#### Libraries and helper functions <a name="libraries"></a>
First, we begin by importing necessary Python libraries and helper functions. These aid in extracting key information from inputs and final outputs.

In [1]:
import os
import re
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)

def print_jsonl_1stline(file):
    with open(file, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        print(json.dumps(data, indent=2))

def filter_files(files, substring):
    for file in files:
        if substring in file:
            return file

def print_jsonl_by_pin(file, pin):
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            if data.get("pin") == pin:
                print(json.dumps(data, indent=2))
                break

[Return to Top](#top)

#### Local paths <a name="paths"></a>
Subsequently, please revise the path variables to ensure they correctly point to the respective paths of the input and output on your machine.

In [2]:
input_folder = 'example/pinned-input'
output_root_path = 'example/consistency-output/synthesized'

[Return to Top](#top)

### 1. Prepare input <a name="inputevents"></a>
The inputs for the synthesis process are pinned heterogeneous collections - in a folder named as "{timestamp}-pin-hc". 

We read from the inputs and display the first pin as an example.

In [3]:
# Print the first pin in the input as an example
batch = [v for v in os.listdir(input_folder) if v.endswith('-pin-hc')][0]
batch_folder = '/'.join([input_folder, batch])
files = [file for file in os.listdir(batch_folder) if file.endswith('.json')]
file = os.path.join(batch_folder, files[0])
print("Sample HC pin:\n")
print_jsonl_1stline(file)

Sample HC pin:

{
  "pin": 2000,
  "events": [
    {
      "asOfDate": 1485993600000,
      "didCreateAsOfDateFromReceivedDate": false,
      "eventData": {
        "booleans": {
          "endMostMessage": false,
          "isNewSubscriberRelation": false
        },
        "complex": {
          "officeOfFiling": "{\"mainCity\":\"Bogota\",\"secondCity\":\"SUESCA\",\"number\":0}"
        },
        "longs": {
          "adjDate": 0,
          "adjective": 0,
          "ageOfMora": 360,
          "arrears": 0,
          "availableValue": 0,
          "balance": 186271,
          "balanceValueInMora": 186271,
          "cardNumber": -1,
          "cellPhone": -1,
          "classCard": 0,
          "clausePermanence": 0,
          "contributionsCancelled": 0,
          "creditLimit": 0,
          "daneCityCodeCorrespondence": -1,
          "daneCityCodeFiling": -1,
          "daneCityCodeLabor": -1,
          "daneCityCodeResidenceOrDaneCityCodeCommercial": -1,
          "dateClausePerm

[Return to Top](#top)

### 2. Define Synthesis extension classes and Lookup <a name="extensions"></a>
The extended classes, such as [SyntheticFieldCustomProcessor](src/main/scala/com/experian/edf/oxygen/consistencycheck/extension/SyntheticFieldCustomProcessor.scala), can be customized for synthesis. 

Similarly, the extended class, like [RelationshipMigrationProcessorImpl](src/main/scala/com/experian/edf/oxygen/consistencycheck/extension/migration/RelationshipMigrationProcessorImpl.scala), can be customized for migration.

In addition, synthesis fields may utilize reference files as lookup tables (e.g., an official national ID list). Here are some sample lookup files:
- [consistency-table1-creditcard.csv](example/lookup/consistency-table1-creditcard.csv)
- [consistencyfiltertable1.csv](example/lookup/consistencyfiltertable1.csv)
- [data_credito_special_case.csv](example/lookup/data_credito_special_case.csv)

[Return to Top](#top)

### 4. Run Synthesis <a name="runsynthesis"></a>
Run Synthesis with sample input, configurations, extended classes, and lookups.

In [4]:
# Uncomment the lines in this block and run pinning
# !cp /data/src/main/resources/log4j.properties /opt/spark/conf/;
# !spark-submit --master local[2] --driver-memory 2g --executor-memory 2g --packages org.apache.spark:spark-avro_2.12:3.3.1 --conf "spark.driver.extraJavaOptions=-Dlog4j.configuration=log4j.properties" --class com.experian.edf.oxygen.consistencycheck.app.SynthesisApp target/oxygen-synthesis-innersource-1.0-jar-with-dependencies.jar env.conf consistency-application.conf oxygen-lookup.conf

[Return to Top](#top)

### 5. Review Synthesis outputs <a name="output"></a>

In [5]:
# Verify output path
output_root_path = 'example/consistency-output/synthesized'
# output_root_path = 'example/output-demo/synthesized'
output_list = os.listdir(output_root_path)
output_path = os.path.join(output_root_path, output_list[0])

# Select an sample event
sample_pin = 2000

In [6]:
# Print one of synthesized pins in the output
files = [file for file in os.listdir(output_path) if file.endswith('.json')]
for f in files[:1]:
    file = os.path.join(output_path, f)
    print("Sample synthesized pin:\n")
    print_jsonl_by_pin(file, sample_pin)

Sample synthesized pin:

{
  "pin": 2000,
  "synthesizedEvents": [
    {
      "asOfPeriod": {
        "epochDateTime": 1485993600000,
        "id": "201702",
        "unit": {
          "name": "MonthOfYear"
        }
      },
      "curatedRecordJson": {
        "ageOfMora": 360,
        "balance": 186271,
        "creditLimit": 0,
        "dueDate": 20301231,
        "firstEventDate": 1485993600000,
        "firstLastName": "TABARES",
        "firstName": "MARTHA",
        "isNewSubscriberRelation": false,
        "middleName": "INES",
        "nationalId": "00020403034",
        "paymentDate": 20060919,
        "refersToRelation": "",
        "secondLastName": "LOPEZ"
      },
      "dataSource": "Oxygen",
      "frequency": {
        "count": 1,
        "unit": {
          "name": "EveryNMonth"
        }
      },
      "migrationHistory": [],
      "parentEventIds": [
        "3140b3bd-f629-6d8d-0000-0010000006e7"
      ],
      "subscriber": "230001",
      "subscriberRelation": 

[Return to Top](#top)

## Conclusion <a name="conclusion"></a>
In conclusion, the Synthesis procedure involves the analysis of pinned records, which are then grouped by periods and account. A singular record is generated for each identified group-period, encompassing fields that represent a reconciliation of the values present within that particular group-period.

[Return to Top](#top)